# Day18 — Team Collaboration + Remote Task Observation

Goal: demonstrate a sanitized, ordered, resumable observation stream for two read-only observers without LLM, Unity, network access, or workflow mutation.

In [ ]:
from datetime import datetime, timezone
import json
import os
import sys
import tempfile

project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()).lower() == 'day18' else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from memory.task_observation import TaskObservationStore
from ui.observation_app import ObservationReader
from workflow.task_observation import TaskObservationProjector

temporary = tempfile.TemporaryDirectory()
database_path = os.path.join(temporary.name, 'workflow.sqlite')
project_id = 'a' * 64
now = '2026-08-20T08:00:00+00:00'
store = TaskObservationStore(database_path, project_id)
projector = TaskObservationProjector(store, project_id, 'local-operator', 'studio-a')
reader = ObservationReader(store, project_id)
print('offline observation store ready')

## Versioned event contract

In [ ]:
projector.project(
    thread_id='thread-1', checkpoint_id='checkpoint-1',
    values={'query': 'private requirement', 'current_agent': 'coder'},
    started_at=now, updated_at=now,
)
projector.project(
    thread_id='thread-1', checkpoint_id='checkpoint-2',
    values={
        'query': 'private requirement', 'current_agent': 'unity_test',
        'code': [{'file': 'Secret.cs', 'content': 'class Secret {}'}],
        'test_result': {'success': True, 'summary': {'total': 2, 'passed': 2}},
    },
    started_at=now, updated_at='2026-08-20T08:01:00+00:00',
)
events = reader.list_events('thread-1', limit=200)
print([(event['cursor'], event['event_type']) for event in events])

## Two observers and disconnect/reconnect

In [ ]:
alice = reader.list_events('thread-1', after_cursor=0, limit=200)
bob_first = reader.list_events('thread-1', after_cursor=0, limit=1)
bob_rest = reader.list_events('thread-1', after_cursor=bob_first[-1]['cursor'], limit=200)
assert alice == bob_first + bob_rest
print({'alice_events': len(alice), 'bob_resumed_from': bob_first[-1]['cursor']})

## Cursor bounds and snapshot reset input

In [ ]:
bounds = reader.cursor_bounds('thread-1')
snapshot = reader.get_snapshot('thread-1')
assert bounds['latest_cursor'] == alice[-1]['cursor']
print({'cursor_bounds': bounds, 'authoritative_gate': snapshot['current_gate']})

## Leakage check

In [ ]:
public_payload = json.dumps(reader.export('thread-1'), ensure_ascii=False)
for forbidden in ('private requirement', 'class Secret', '"query"', '"code"', '"diff"'):
    assert forbidden not in public_payload
print('sanitized export passed')

## Next Steps

Run the focused Day18 tests, the complete offline suite, and a separately documented LAN browser probe. Remote approval, cancellation, retry, Git operations, and task takeover remain disabled.